# IQL with a negative policy — walker2d

Reproduces the Group 13 study. Run the cells in order.

Four algorithms are compared, all on `mujoco/walker2d/medium-v0`:

| | |
|---|---|
| **IQL (standard)** | ordinary IQL, no negative policy — the baseline |
| **IQL pessimistic** | plain IQL with a lower expectile, the cheap alternative |
| **IQL + negative [awr]** | our method, danger folded into the imitation weight |
| **IQL + negative [dist]** | our method, the TD3+BC penalty ported over |

`SEEDS` in the sweep cell is the knob that matters: **5 seeds says the negative
policy wins, 15 seeds says it does not.** That reversal is the finding.

The networks are tiny, so CPU and GPU are closer than you would expect.
There is a benchmark cell below that measures both on your runtime and
picks the faster one; attach a T4 GPU first if you want it compared.

In [ ]:
!pip install -q minari "gymnasium[mujoco]" h5py

In [ ]:
import os, math, copy, json, time, random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from torch.optim.lr_scheduler import CosineAnnealingLR

import gymnasium as gym
import minari

torch.set_num_threads(2)
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

In [ ]:
# ---------------------------------------------------------------------------
# The dataset. Minari hosts the same locomotion data as D4RL, which needs
# Python 3.9 and mujoco-py and will not install on a current runtime.
# We use `medium` because it is the only walker2d set with a real tail of
# failures (5.6% of episodes end in a fall). On `expert` there is nothing to
# improve and no unsafe behaviour for a danger policy to learn from.
# First run downloads ~200 MB.
# ---------------------------------------------------------------------------
DATASET = "mujoco/walker2d/medium-v0"

ds = minari.load_dataset(DATASET, download=True)
ENV_ID = ds.env_spec.id

obs, act, rew, nobs, term, ep_len = [], [], [], [], [], []
for ep in ds.iterate_episodes():
    o = np.asarray(ep.observations, dtype=np.float32)      # (T+1, obs_dim)
    obs.append(o[:-1]); nobs.append(o[1:])
    act.append(np.asarray(ep.actions, dtype=np.float32))
    rew.append(np.asarray(ep.rewards, dtype=np.float32))
    term.append(np.asarray(ep.terminations, dtype=bool))   # terminations only
    ep_len.append(len(ep.actions))

DATA = {
    "observations":      np.concatenate(obs, 0),
    "actions":           np.concatenate(act, 0),
    "rewards":           np.concatenate(rew, 0),
    "next_observations": np.concatenate(nobs, 0),
    "terminals":         np.concatenate(term, 0).astype(np.float32),
    "episode_ends":      np.cumsum(ep_len) - 1,
}

ep_returns = np.array([r.sum() for r in rew])
falls = np.array([t.any() for t in term])
print(f"env         {ENV_ID}")
print(f"episodes    {len(ep_returns):,}    transitions {len(DATA['rewards']):,}")
print(f"obs / act   {DATA['observations'].shape[1]} / {DATA['actions'].shape[1]}")
print(f"return      mean {ep_returns.mean():.1f}   median {np.median(ep_returns):.1f}"
      f"   p1 {np.percentile(ep_returns, 1):.1f}   min {ep_returns.min():.1f}")
print(f"falls       {falls.sum()} of {len(falls)} episodes ({100 * falls.mean():.1f}%)")

In [ ]:
# ---------------------------------------------------------------------------
# Normalised score. Raw walker2d returns are ~6000 because
#     return = steps per episode (up to 1000) x reward per step (~6.1),
# which is not comparable with Pendulum's -162. So we report
#     score = 100 * (return - random) / (expert - random)
# where 0 = a random policy and 100 = the expert dataset. A LOW SCORE MEANS THE
# AGENT FELL EARLY, not that it walked badly.
# The reference points are fixed constants so this notebook, the slides and the
# report give identical numbers; the measurement below is only a check.
# ---------------------------------------------------------------------------
RANDOM_RETURN = 2.1      # Walker2d-v5, mean over 30 random episodes
EXPERT_RETURN = 6847.8   # mean return of Minari mujoco/walker2d/expert-v0

def normalised(x):
    return 100.0 * (x - RANDOM_RETURN) / (EXPERT_RETURN - RANDOM_RETURN)

def random_policy_return(env_id, n=20, seed=5000):
    env = gym.make(env_id)
    rets, lens = [], []
    for i in range(n):
        o, _ = env.reset(seed=seed + i)
        done, tot, steps = False, 0.0, 0
        while not done:
            o, r, tm, tr, _ = env.step(env.action_space.sample())
            tot += r; steps += 1; done = tm or tr
        rets.append(tot); lens.append(steps)
    env.close()
    return float(np.mean(rets)), float(np.mean(lens))

measured, mlen = random_policy_return(ENV_ID)
print(f"random policy, measured here : {measured:7.1f}  over {mlen:.0f} steps")
print(f"reference used for the score : {RANDOM_RETURN:7.1f}  -> score {normalised(RANDOM_RETURN):5.1f}")
print(f"dataset behaviour policy     : {ep_returns.mean():7.1f}  -> score {normalised(ep_returns.mean()):5.1f}")
print(f"expert data                  : {EXPERT_RETURN:7.1f}  -> score {normalised(EXPERT_RETURN):5.1f}")

In [ ]:
# --------------------------- networks (IQL reference) ----------------------
EXP_ADV_MAX = 100.0
LOG_STD_MIN, LOG_STD_MAX = -20.0, 2.0


class Squeeze(nn.Module):
    def __init__(self, dim=-1):
        super().__init__(); self.dim = dim
    def forward(self, x): return x.squeeze(dim=self.dim)


class MLP(nn.Module):
    def __init__(self, dims, out_act=None, squeeze=False):
        super().__init__()
        layers = []
        for i in range(len(dims) - 2):
            layers += [nn.Linear(dims[i], dims[i + 1]), nn.ReLU()]
        layers.append(nn.Linear(dims[-2], dims[-1]))
        if out_act is not None: layers.append(out_act())
        if squeeze: layers.append(Squeeze(-1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)


class GaussianPolicy(nn.Module):
    def __init__(self, sd, ad, max_action, hidden=256, n_hidden=2):
        super().__init__()
        self.net = MLP([sd, *([hidden] * n_hidden), ad], out_act=nn.Tanh)
        self.log_std = nn.Parameter(torch.zeros(ad, dtype=torch.float32))
        self.max_action = max_action
    def forward(self, obs):
        return Normal(self.net(obs), torch.exp(self.log_std.clamp(LOG_STD_MIN, LOG_STD_MAX)))
    @torch.no_grad()
    def act(self, state, device):
        s = torch.tensor(state.reshape(1, -1), device=device, dtype=torch.float32)
        d = self(s)
        a = d.mean if not self.training else d.sample()
        return torch.clamp(self.max_action * a, -self.max_action,
                           self.max_action).cpu().numpy().flatten()


class TwinQ(nn.Module):
    def __init__(self, sd, ad, hidden=256, n_hidden=2):
        super().__init__()
        dims = [sd + ad, *([hidden] * n_hidden), 1]
        self.q1, self.q2 = MLP(dims, squeeze=True), MLP(dims, squeeze=True)
    def both(self, s, a):
        sa = torch.cat([s, a], 1); return self.q1(sa), self.q2(sa)
    def forward(self, s, a): return torch.min(*self.both(s, a))


class ValueFunction(nn.Module):
    def __init__(self, sd, hidden=256, n_hidden=2):
        super().__init__()
        self.v = MLP([sd, *([hidden] * n_hidden), 1], squeeze=True)
    def forward(self, s): return self.v(s)


def soft_update(target, source, tau):
    for tp, sp in zip(target.parameters(), source.parameters()):
        tp.data.copy_((1 - tau) * tp.data + tau * sp.data)


def asymmetric_l2_loss(u, tau):
    # The expectile loss. tau > 0.5 pulls V towards HIGH Q (optimistic);
    # tau < 0.5 pulls it towards low Q (pessimistic).
    return torch.mean(torch.abs(tau - (u < 0).float()) * u ** 2)


class ReplayBuffer:
    def __init__(self, data, device):
        t = lambda x: torch.tensor(x, dtype=torch.float32, device=device)
        self.states, self.actions = t(data["observations"]), t(data["actions"])
        self.rewards, self.dones = t(data["rewards"]), t(data["terminals"])
        self.next_states = t(data["next_observations"])
        self.size, self.device = self.states.shape[0], device
    def sample(self, n):
        i = torch.randint(0, self.size, (n,), device=self.device)
        return (self.states[i], self.actions[i], self.rewards[i],
                self.next_states[i], self.dones[i])

In [ ]:
# ---------------------------------------------------------------------------
# IQL, plus the negative policy.
#
# IQL does NOT maximise Q through the actor the way TD3+BC does: its policy step
# is advantage-weighted regression, a weighted imitation of the dataset. So the
# TD3+BC penalty does not carry over directly, and there are two options:
#
#   neg_mode="awr"   the danger advantage goes into the imitation weight
#                        stock IQL:  w = exp( beta*A )
#                        ours:       w = exp( beta*A - beta_neg*A_neg )
#                    no second actor at all.
#
#   neg_mode="dist"  the direct port: train a danger actor, push the main actor
#                    away from it. The draft's penalty is unbounded, so here it
#                    is a hinge, max(0, margin - d), which stops pushing once far
#                    enough away.
#
#   neg_mode="none"  ordinary IQL. THIS IS THE BASELINE everything is compared to.
#
# Why A_neg is not just -A: V = expectile(Q, tau=0.7) is an UPPER expectile, so
# V_neg = expectile(-Q, 0.7) = -expectile(Q, 0.3), a LOWER one. The two stacks
# bootstrap through different continuations -- Q assumes you act well afterwards,
# -Q_neg assumes you act badly. That gap is the risk signal.
# ---------------------------------------------------------------------------
class ImplicitQLearning:
    def __init__(self, sd, ad, max_action, device, max_steps,
                 beta=3.0, iql_tau=0.7, discount=0.99, tau=0.005, lr=3e-4,
                 neg_mode="none", beta_neg=0.0, iql_tau_neg=0.7, margin=0.25):
        self.device, self.max_action = device, max_action
        self.beta, self.iql_tau, self.discount, self.tau = beta, iql_tau, discount, tau
        self.neg_mode, self.beta_neg = neg_mode, beta_neg
        self.iql_tau_neg, self.margin = iql_tau_neg, margin
        self.use_neg = neg_mode != "none"

        self.qf = TwinQ(sd, ad).to(device)
        self.q_target = copy.deepcopy(self.qf).requires_grad_(False).to(device)
        self.vf = ValueFunction(sd).to(device)
        self.actor = GaussianPolicy(sd, ad, max_action).to(device)
        self.q_opt = torch.optim.Adam(self.qf.parameters(), lr=lr)
        self.v_opt = torch.optim.Adam(self.vf.parameters(), lr=lr)
        self.a_opt = torch.optim.Adam(self.actor.parameters(), lr=lr)
        self.a_sched = CosineAnnealingLR(self.a_opt, max_steps)

        if self.use_neg:
            self.qf_neg = TwinQ(sd, ad).to(device)
            self.q_neg_target = copy.deepcopy(self.qf_neg).requires_grad_(False).to(device)
            self.vf_neg = ValueFunction(sd).to(device)
            self.q_neg_opt = torch.optim.Adam(self.qf_neg.parameters(), lr=lr)
            self.v_neg_opt = torch.optim.Adam(self.vf_neg.parameters(), lr=lr)
        if neg_mode == "dist":
            self.neg_actor = GaussianPolicy(sd, ad, max_action).to(device)
            self.neg_a_opt = torch.optim.Adam(self.neg_actor.parameters(), lr=lr)

    def _update_v(self, vf, q_target, obs, act, iql_tau, opt):
        with torch.no_grad():
            target_q = q_target(obs, act)
        adv = target_q - vf(obs)
        loss = asymmetric_l2_loss(adv, iql_tau)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        return adv

    def _update_q(self, qf, q_target, next_v, obs, act, rew, dones, opt):
        # next_v is from BEFORE this step's value update, as in the reference
        targets = rew + (1.0 - dones) * self.discount * next_v
        qs = qf.both(obs, act)
        loss = sum(F.mse_loss(q, targets) for q in qs) / len(qs)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        soft_update(q_target, qf, self.tau)

    def train(self, batch):
        obs, act, rew, next_obs, dones = batch
        with torch.no_grad():
            next_v = self.vf(next_obs)
            next_v_neg = self.vf_neg(next_obs) if self.use_neg else None

        adv = self._update_v(self.vf, self.q_target, obs, act, self.iql_tau, self.v_opt)
        self._update_q(self.qf, self.q_target, next_v, obs, act, rew, dones, self.q_opt)

        adv_neg = None
        if self.use_neg:
            adv_neg = self._update_v(self.vf_neg, self.q_neg_target, obs, act,
                                     self.iql_tau_neg, self.v_neg_opt)
            self._update_q(self.qf_neg, self.q_neg_target, next_v_neg, obs, act,
                           -rew, dones, self.q_neg_opt)       # danger reward = -r
            if self.neg_mode == "dist":
                w = torch.exp((self.beta * adv_neg.detach()).clamp(max=math.log(EXP_ADV_MAX)))
                w = w / (w.mean() + 1e-8)
                bc = -self.neg_actor(obs).log_prob(act).sum(-1)
                loss = torch.mean(w * bc)
                self.neg_a_opt.zero_grad(set_to_none=True)
                loss.backward(); self.neg_a_opt.step()

        logits = self.beta * adv.detach()
        if self.neg_mode == "awr":
            # adv_neg is near 0 for the most dangerous in-support actions and very
            # negative for safe ones, so subtracting it upweights the safe ones
            logits = logits - self.beta_neg * adv_neg.detach()
        w = torch.exp(logits.clamp(max=math.log(EXP_ADV_MAX)))
        # hold the mean weight at 1, so changing beta_neg changes WHICH actions
        # are favoured rather than the effective learning rate
        w = w / (w.mean() + 1e-8)

        dist = self.actor(obs)
        policy_loss = torch.mean(w * (-dist.log_prob(act).sum(-1)))

        if self.neg_mode == "dist":
            with torch.no_grad():
                mu_neg = self.neg_actor(obs).mean
            d = ((dist.mean - mu_neg) ** 2).mean(dim=-1)
            policy_loss = policy_loss + self.beta_neg * torch.clamp(self.margin - d, min=0.0).mean()

        self.a_opt.zero_grad(set_to_none=True); policy_loss.backward()
        self.a_opt.step(); self.a_sched.step()

In [ ]:
# --------------------------- CPU or GPU? -----------------------------------
# Not obvious here. The networks are two 256-unit layers, so each training step
# launches many tiny kernels: whoever has the lower per-operation overhead wins,
# not whoever has more compute. On Apple silicon the CPU beats MPS by 4-5x; on
# Colab's CUDA the GPU usually wins, because its launch overhead is far smaller
# and Colab's CPU is only 2 vCPU. Twenty seconds here settles it for YOUR runtime.
#
# Attach a GPU first if you want the comparison:
#     Runtime -> Change runtime type -> T4 GPU
# Then set DEVICE below to whichever wins and re-run the sweep.
def benchmark(device, n_steps=150, neg_mode="awr"):
    sub = {k: DATA[k][:50_000] for k in
           ("observations", "actions", "rewards", "next_observations", "terminals")}
    sd, ad = sub["observations"].shape[1], sub["actions"].shape[1]
    buf = ReplayBuffer(sub, device)
    agent = ImplicitQLearning(sd, ad, 1.0, device, max_steps=n_steps, neg_mode=neg_mode)
    for _ in range(20):                       # warm up
        agent.train(buf.sample(256))
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_steps):
        agent.train(buf.sample(256))
    if device == "cuda": torch.cuda.synchronize()
    return n_steps / (time.time() - t0)


devices = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
rates = {}
for dev in devices:
    rates[dev] = benchmark(dev)
    print(f"  {dev:4s}  {rates[dev]:6.1f} steps/s   ->  25,000 steps in "
          f"{25_000 / rates[dev] / 60:5.1f} min per run")

best = max(rates, key=rates.get)
print(f"\nfaster here: {best}")
if len(devices) == 1:
    print("No GPU attached -- Runtime -> Change runtime type -> T4 GPU to compare.")

DEVICE = best      # the sweep below uses this

In [ ]:
# --------------------------- evaluation and one run ------------------------
# The worst case is a property of the return DISTRIBUTION, so evaluation keeps
# every episode rather than a summary.
@torch.no_grad()
def evaluate(env, actor, device, n_episodes, seed, state_mean, state_std):
    actor.eval()
    returns, fell = [], []
    for i in range(n_episodes):
        o, _ = env.reset(seed=seed + 10_000 + i)
        done, total, terminated = False, 0.0, False
        while not done:
            a = actor.act((o - state_mean) / state_std, device)
            o, r, terminated, truncated, _ = env.step(a)
            total += r; done = terminated or truncated
        returns.append(total); fell.append(float(terminated))
    actor.train()
    return np.asarray(returns), np.asarray(fell)


def cvar(x, alpha=0.10):
    # mean of the worst alpha-fraction: a stable left-tail statistic. A bare
    # minimum over a finite sample is what misled us at 5 seeds.
    k = max(1, int(math.ceil(alpha * len(x))))
    return float(np.sort(x)[:k].mean())


def run(seed=0, steps=25_000, neg_mode="none", beta_neg=0.0, iql_tau=0.7,
        n_episodes=50, batch_size=256, device=None, verbose=True):
    device = device or globals().get("DEVICE") or (
        "cuda" if torch.cuda.is_available() else "cpu")
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    data = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in DATA.items()}

    # CORL's locomotion reward scaling: map the dataset's return range onto ~[0,1000]
    starts = np.r_[0, data["episode_ends"][:-1] + 1]
    rets = np.add.reduceat(data["rewards"], starts)
    data["rewards"] = data["rewards"] / (rets.max() - rets.min()) * 1000.0

    state_mean = data["observations"].mean(0)
    state_std = data["observations"].std(0) + 1e-3
    data["observations"] = (data["observations"] - state_mean) / state_std
    data["next_observations"] = (data["next_observations"] - state_mean) / state_std

    env = gym.make(ENV_ID)
    sd, ad = data["observations"].shape[1], data["actions"].shape[1]
    buf = ReplayBuffer(data, device)
    agent = ImplicitQLearning(sd, ad, float(env.action_space.high[0]), device,
                              max_steps=steps, iql_tau=iql_tau,
                              neg_mode=neg_mode, beta_neg=beta_neg)

    t0 = time.time()
    for _ in range(steps):
        agent.train(buf.sample(batch_size))

    returns, fell = evaluate(env, agent.actor, device, n_episodes, seed,
                             state_mean, state_std)
    env.close()
    s = {"mean": float(returns.mean()), "median": float(np.median(returns)),
         "worst": float(returns.min()), "best": float(returns.max()),
         "cvar10": cvar(returns, 0.10), "fall_rate": float(fell.mean()),
         "seed": seed, "neg_mode": neg_mode, "beta_neg": beta_neg,
         "iql_tau": iql_tau, "returns": returns.tolist(),
         "seconds": time.time() - t0}
    if verbose:
        print(f"    seed {seed}  score {normalised(s['mean']):5.1f}   "
              f"worst {normalised(s['worst']):5.1f}   "
              f"CVaR10 {normalised(s['cvar10']):5.1f}   "
              f"falls {100 * s['fall_rate']:3.0f}%   [{s['seconds']:4.0f}s]")
    return s

In [ ]:
# --------------------------- the sweep -------------------------------------
# SEEDS is the knob that matters:
#     5 seeds  -> the negative policy looks like a clear win   (~1h15 on CPU)
#    15 seeds  -> the advantage disappears                     (~4h on CPU)
# Seeds are independent, so you can run 5 now and extend to 15 later.
SEEDS      = 5
STEPS      = 25_000     # IQL has essentially converged on this dataset by 25k
N_EPISODES = 50         # the report used 100; more episodes = a better tail estimate

CONFIGS = [
    # ordinary IQL, no negative policy -- the baseline everything is compared to
    dict(label="IQL (standard)",           neg_mode="none", beta_neg=0.0, iql_tau=0.7),
    # plain IQL made pessimistic by one number: the cheap alternative to beat
    dict(label="IQL pessimistic, tau=0.5", neg_mode="none", beta_neg=0.0, iql_tau=0.5),
    # our method, both variants
    dict(label="IQL + negative [awr]",     neg_mode="awr",  beta_neg=0.5, iql_tau=0.7),
    dict(label="IQL + negative [dist]",    neg_mode="dist", beta_neg=0.5, iql_tau=0.7),
]

results = defaultdict(list)
for cfg in CONFIGS:
    print(f"\n{cfg['label']}")
    kw = {k: v for k, v in cfg.items() if k != "label"}
    for seed in range(SEEDS):
        results[cfg["label"]].append(
            run(seed=seed, steps=STEPS, n_episodes=N_EPISODES, **kw))

with open("iql_results.json", "w") as f:
    json.dump(dict(results), f)
print("\nsaved -> iql_results.json")

In [ ]:
# --------------------------- the results table -----------------------------
def paired_bootstrap(base, alt, n=20000, seed=0):
    rng = np.random.default_rng(seed)
    k = min(len(base), len(alt))
    if k == 0: return float("nan")
    idx = rng.integers(0, k, size=(n, k))
    return float((np.asarray(alt)[idx].mean(1) - np.asarray(base)[idx].mean(1) > 0).mean())


def table(results):
    labels = [c["label"] for c in CONFIGS if c["label"] in results]
    base = labels[0]
    keys = ("mean", "median", "worst", "best", "cvar10", "fall_rate")
    agg = {l: {k: np.array([r[k] for r in results[l]]) for k in keys} for l in labels}

    print(f"IQL on walker2d-medium  "
          f"({len(results[base])} seeds x {N_EPISODES} episodes, {STEPS:,} steps)")
    print(f"Normalised score: 0 = random policy, 100 = expert data.\n")
    head = f"{'Configuration':<26}{'Mean':>9}{'Median':>9}{'Worst':>9}{'Best':>9}"
    print(head); print("-" * len(head))
    for l in labels:
        a = agg[l]
        print(f"{l:<26}{normalised(a['mean'].mean()):>9.1f}"
              f"{normalised(a['median'].mean()):>9.1f}"
              f"{normalised(a['worst'].mean()):>9.1f}"
              f"{normalised(a['best'].mean()):>9.1f}")
    print("\nWorst = per seed the minimum over its episodes, averaged over seeds.")

    print("\nSupporting detail (not for the report table):")
    for l in labels:
        a = agg[l]
        p = "-" if l == base else f"{paired_bootstrap(agg[base]['cvar10'], a['cvar10']):.2f}"
        sd = normalised(a["cvar10"].std()) - normalised(0.0)
        print(f"  {l:<26} CVaR@10 {normalised(a['cvar10'].mean()):5.1f} +- {sd:<5.1f}"
              f"  falls {100 * a['fall_rate'].mean():3.0f}%"
              f"  P(better than baseline) {p}")

table(results)

In [ ]:
# --------------------------- where the result comes from -------------------
# Look at the spread, not the averages. At 5 seeds the baseline usually contains
# one catastrophic seed, and that single seed is what makes the negative policy
# look good. Our 15-seed baseline came out as
#
#   15.0, 59.1, 72.1, 73.5, 76.2, 81.0, 81.4, 82.1,
#   84.7, 85.9, 87.1, 87.1, 87.4, 87.6, 87.9
#   ^^^^ landed inside the first five
#
# so the first 5 seeds averaged 67.7 while all 15 average 76.5.
for cfg in CONFIGS:
    l = cfg["label"]
    if l not in results: continue
    v = np.sort([normalised(r["cvar10"]) for r in results[l]])
    print(f"{l:<26} " + ", ".join(f"{x:.1f}" for x in v))
    print(f"{'':<26} mean {v.mean():.1f}   sd {v.std():.1f}\n")